In [26]:
import os
import json
from pathlib import Path
from datetime import datetime
import numpy as np
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
 
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split
from torch.amp import GradScaler, autocast
from torchvision import transforms
from PIL import Image
import timm

from dataset import PlantWildDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

2.10.0+cu128
True
NVIDIA GeForce RTX 3060 Laptop GPU


In [8]:
# Extract image embeddings
image_data = torch.load("./checkpoints/image_embeddings.pt")
img_train_embeddings = image_data["train_embeddings"]   
img_train_labels     = image_data["train_labels"]       
img_test_embeddings  = image_data["test_embeddings"]    
img_test_labels      = image_data["test_labels"]        

# Extract text features
text_data  = torch.load("./checkpoints/bert_features.pt")
txt_train_embeddings = text_data["train_features"]   
txt_train_labels     = text_data["train_labels"]       
txt_test_embeddings  = text_data["test_features"]    
txt_test_labels      = text_data["test_labels"] 

print(f"Image train embeddings shape:   {img_train_embeddings.shape}")
print(f"Text train embeddings shape:    {txt_train_embeddings.shape}")
print(f"Image test embeddings shape:    {img_test_embeddings.shape}")
print(f"Text test embeddings shape:     {txt_test_embeddings.shape}")
print(f"Image train labels shape:       {img_train_labels.shape}")
print(f"Text train labels shape:        {txt_train_labels.shape}")
print(f"Image test labels shape:        {img_test_labels.shape}")
print(f"Text test labels shape:         {txt_test_labels.shape}")

Image train embeddings shape:   torch.Size([14832, 768])
Text train embeddings shape:    torch.Size([3560, 768])
Image test embeddings shape:    torch.Size([3708, 768])
Text test embeddings shape:     torch.Size([890, 768])
Image train labels shape:       torch.Size([14832])
Text train labels shape:        torch.Size([3560])
Image test labels shape:        torch.Size([3708])
Text test labels shape:         torch.Size([890])


In [9]:
# check if labels overlap
print(txt_train_labels[:10])
print(img_train_labels[:10])

# check unique classes in each
print(f"Unique image classes: {img_train_labels.unique().shape}")
print(f"Unique text classes:  {txt_train_labels.unique().shape}")

tensor([19, 18, 27, 87, 11, 51,  0, 71, 11, 49])
tensor([70, 63, 69, 26, 84, 48, 73, 20, 71, 48])
Unique image classes: torch.Size([89])
Unique text classes:  torch.Size([89])


In [28]:
def align_embeddings(img_emb, img_lbl, txt_emb, txt_lbl):
    aligned_img, aligned_txt, aligned_lbl = [], [], []

    for class_idx in range(89):
        img_mask  = (img_lbl == class_idx)
        txt_mask  = (txt_lbl == class_idx)
        img_class = img_emb[img_mask]
        txt_class = txt_emb[txt_mask]

        if len(txt_class) == 0:
            continue

        n = len(img_class)   # keep ALL image samples
        # randomly sample text embeddings with replacement
        repeat_idx = torch.randint(0, len(txt_class), (n,))
        aligned_img.append(img_class)
        aligned_txt.append(txt_class[repeat_idx])
        aligned_lbl.append(torch.full((n,), class_idx, dtype=torch.long))

    return (torch.cat(aligned_img), torch.cat(aligned_txt), torch.cat(aligned_lbl))

train_img, train_txt, train_lbl = align_embeddings(
    img_train_embeddings, img_train_labels,
    txt_train_embeddings, txt_train_labels
)
test_img, test_txt, test_lbl = align_embeddings(
    img_test_embeddings, img_test_labels,
    txt_test_embeddings, txt_test_labels
)

print(f"Aligned train: img={train_img.shape}, txt={train_txt.shape}, lbl={train_lbl.shape}")
print(f"Aligned test:  img={test_img.shape},  txt={test_txt.shape},  lbl={test_lbl.shape}")

Aligned train: img=torch.Size([14832, 768]), txt=torch.Size([14832, 768]), lbl=torch.Size([14832])
Aligned test:  img=torch.Size([3708, 768]),  txt=torch.Size([3708, 768]),  lbl=torch.Size([3708])


In [19]:
train_dataset = TensorDataset(train_img, train_txt, train_lbl)
test_dataset  = TensorDataset(test_img,  test_txt,  test_lbl)

train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [20]:
class MultimodalMLP(nn.Module):
    """
    Fuses image and text embeddings and classifies plant disease.

    image_emb (768) ─┐
                      ├→ concat (1536) → MLP → (89 classes)
    text_emb  (768) ─┘
    """

    def __init__(self, image_dim=768, text_dim=768,
                 num_classes=89, dropout=0.3):
        super().__init__()

        fused_dim = image_dim + text_dim   # 1536

        self.mlp = nn.Sequential(
            nn.Linear(fused_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, num_classes),
        )

    def forward(self, image_emb, text_emb):
        fused = torch.cat([image_emb, text_emb], dim=1)  # (B, 1536)
        return self.mlp(fused)                            # (B, 89)

In [21]:
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS     = 50
LR         = 1e-3
NUM_CLASSES = 89
MLP_SAVE  = "./checkpoints/best_multimodal_mlp.pt"

model     = MultimodalMLP(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2
)

In [23]:
torch.cuda.manual_seed(42)
torch.manual_seed(42)

best_acc  = 0.0
for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for img_emb, txt_emb, labels in train_loader:
        img_emb, txt_emb, labels = (img_emb.to(DEVICE),
                                    txt_emb.to(DEVICE),
                                    labels.to(DEVICE))
        optimizer.zero_grad()
        loss = criterion(model(img_emb, txt_emb), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step(epoch)

    # ── Evaluate ───────────────────────────────────────────────
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for img_emb, txt_emb, labels in test_loader:
            img_emb, txt_emb, labels = (img_emb.to(DEVICE),
                                        txt_emb.to(DEVICE),
                                        labels.to(DEVICE))
            preds = model(img_emb, txt_emb).argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch {epoch:>3}/{EPOCHS}  "
          f"Loss: {train_loss / len(train_loader):.4f}  "
          f"Test Acc: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), MLP_SAVE)
        print(f"  ✓ Best MLP saved ({best_acc:.2f}%)")

print(f"\nDone — best Test Acc: {best_acc:.2f}%")

Epoch   1/50  Loss: 0.7722  Test Acc: 74.70%
  ✓ Best MLP saved (74.70%)
Epoch   2/50  Loss: 0.7824  Test Acc: 74.27%
Epoch   3/50  Loss: 0.7843  Test Acc: 74.00%
Epoch   4/50  Loss: 0.7801  Test Acc: 74.65%
Epoch   5/50  Loss: 0.7783  Test Acc: 74.54%
Epoch   6/50  Loss: 0.7777  Test Acc: 74.49%
Epoch   7/50  Loss: 0.7767  Test Acc: 74.49%
Epoch   8/50  Loss: 0.7752  Test Acc: 74.33%
Epoch   9/50  Loss: 0.7756  Test Acc: 74.60%
Epoch  10/50  Loss: 0.7745  Test Acc: 74.54%
Epoch  11/50  Loss: 0.7739  Test Acc: 74.38%
Epoch  12/50  Loss: 0.7734  Test Acc: 74.49%
Epoch  13/50  Loss: 0.7726  Test Acc: 74.60%
Epoch  14/50  Loss: 0.7721  Test Acc: 74.60%
Epoch  15/50  Loss: 0.7720  Test Acc: 74.73%
  ✓ Best MLP saved (74.73%)
Epoch  16/50  Loss: 0.7718  Test Acc: 74.70%
Epoch  17/50  Loss: 0.7716  Test Acc: 74.70%
Epoch  18/50  Loss: 0.7715  Test Acc: 74.65%
Epoch  19/50  Loss: 0.7715  Test Acc: 74.65%
Epoch  20/50  Loss: 0.7715  Test Acc: 74.68%
Epoch  21/50  Loss: 0.7797  Test Acc: 74.06%

In [27]:
# ── Evaluate ───────────────────────────────────────────────
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for img_emb, txt_emb, labels in test_loader:
        img_emb, txt_emb, labels = (img_emb.to(DEVICE),
                                    txt_emb.to(DEVICE),
                                    labels.to(DEVICE))
        preds = model(img_emb, txt_emb).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc       = 100 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
precision = precision_score(all_labels, all_preds, average="macro", zero_division=0) * 100
recall    = recall_score(all_labels, all_preds, average="macro", zero_division=0) * 100
f1        = f1_score(all_labels, all_preds, average="macro", zero_division=0) * 100

print(f"Epoch {epoch:>3}/{EPOCHS}  "
      f"Loss: {train_loss / len(train_loader):.4f}  "
      f"Acc: {acc:.2f}%  "
      f"Precision: {precision:.2f}%  "
      f"Recall: {recall:.2f}%  "
      f"F1: {f1:.2f}%")

Epoch  50/50  Loss: 0.7709  Acc: 74.51%  Precision: 75.27%  Recall: 74.49%  F1: 73.67%
